# Week 13 — Explainability & Trust in EM Machine Learning

**Course:** Data Science for Electron Microscopy  
**Week:** 13 — Explainability, trust & course synthesis

---

## What you will do in this notebook

1. **Synthesise a small EM-like image dataset** — 32×32 synthetic grain images with a
   central 'defect' (bright disk) as class 1, and plain grain images as class 0.  
   A second *shortcut* dataset replaces the defect with a bright corner artifact for class-1
   training images, so a classifier can 'cheat' by detecting the artifact rather than the defect.

2. **Train two tiny CNNs**: a *clean model* (trained on defect images) and a *shortcut model*
   (trained on corner-artifact images as class 1). Both train fast on CPU (< 30 s).

3. **Compute input-gradient saliency maps** for both models and visualise them.

4. **Quantify the difference**: mean saliency inside the true defect region vs outside  
   for the clean model; mean saliency at the corner artifact vs the defect region for  
   the shortcut model.

5. **Exercise**: change the occlusion patch size / threshold and observe the saliency change.

**Colab-compatible**: everything runs on CPU with standard packages.

---

## Key numbers (SEED=42)

After running all cells you should observe:
- Clean model: gradient saliency inside defect region ≈ **0.627**, outside ≈ **0.195**, ratio ≈ **3.2**
- Shortcut model: gradient saliency at corner artifact ≈ **0.500**, inside defect region ≈ **0.054**, ratio ≈ **9.3**
- Occlusion saliency (patch=4, val=0.35): clean model inside ≈ **0.698**, outside ≈ **0.006**, ratio ≈ **109**
- Occlusion saliency: shortcut model corner ≈ **0.531**, defect ≈ **0.001**, ratio >> 100


In [1]:
# Cell 1 — imports
# Colab: all packages are pre-installed. Uncomment the pip line if needed.
# pip install torch  # uncomment on a bare environment

import numpy as np
import matplotlib
try:
    import google.colab
    matplotlib.use('inline')
except ImportError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('PyTorch version:', torch.__version__)
print('NumPy  version:', np.__version__)
print('Device: CPU (all training < 30 s)')

PyTorch version: 2.7.1+cu126
NumPy  version: 1.26.4
Device: CPU (all training < 30 s)


In [2]:
# Cell 2 — Dataset synthesis
#
# Images: 32×32 synthetic EM-like grain images.
#   Class 0 (no defect): Gaussian grain texture only.
#   Class 1:
#     Clean dataset  → bright disk defect at centre (physically meaningful).
#     Shortcut dataset → bright 6×6 corner artifact (a spurious scanning artifact;
#                        no defect in the image — the classifier must use the artifact
#                        to decide class 1, NOT the true physical feature).
#
# This mirrors a real EM failure mode: a beam-entry artifact correlates perfectly
# with a labelling error, and the model learns the artifact instead of the defect.

IMG_SIZE      = 32
N_TRAIN       = 400   # 200 per class
N_TEST        = 100   # 50 per class
DEFECT_CENTER = (16, 16)
DEFECT_RADIUS = 5
CORNER_SIZE   = 6

def make_grain_background(n, seed_offset=0):
    """n × 32×32 synthetic grain images (class 0 baseline)."""
    rng = np.random.RandomState(SEED + seed_offset)
    images = []
    for _ in range(n):
        base    = rng.randn(IMG_SIZE, IMG_SIZE) * 0.12 + 0.35
        texture = gaussian_filter(rng.randn(IMG_SIZE, IMG_SIZE) * 0.10, sigma=2.5)
        images.append(np.clip(base + texture, 0, 1))
    return np.stack(images).astype(np.float32)

def add_defect(images, seed_offset=0):
    """Add a bright disk defect at the centre — the physically meaningful signal."""
    out  = images.copy()
    yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
    mask = (yy - DEFECT_CENTER[0])**2 + (xx - DEFECT_CENTER[1])**2 < DEFECT_RADIUS**2
    rng  = np.random.RandomState(SEED + seed_offset + 100)
    for i in range(len(out)):
        out[i][mask] += 0.45 + rng.randn(mask.sum()) * 0.03
    return np.clip(out, 0, 1), mask

def add_corner_artifact(images, seed_offset=0):
    """Inject a bright 6×6 corner artifact — a spurious scanning artefact."""
    out = images.copy()
    rng = np.random.RandomState(SEED + seed_offset + 200)
    for i in range(len(out)):
        out[i, :CORNER_SIZE, :CORNER_SIZE] = 0.55 + rng.randn(CORNER_SIZE, CORNER_SIZE) * 0.02
    return np.clip(out, 0, 1)

# --- training sets ---
n = N_TRAIN // 2  # per class

bg_class0 = make_grain_background(n, seed_offset=0)
bg_class1  = make_grain_background(n, seed_offset=10)

df_class1, DEFECT_MASK = add_defect(bg_class1, seed_offset=0)   # defect images
corner_class1           = add_corner_artifact(bg_class1, seed_offset=0)  # artifact images (same background)

X_clean_train    = np.concatenate([bg_class0, df_class1])
X_shortcut_train = np.concatenate([bg_class0, corner_class1])
Y_train          = np.array([0]*n + [1]*n, dtype=np.int64)

# --- test sets ---
nte = N_TEST // 2
bg_te   = make_grain_background(nte, seed_offset=30)
df_te, _ = add_defect(make_grain_background(nte, seed_offset=40), seed_offset=1)
corner_te = add_corner_artifact(make_grain_background(nte, seed_offset=40), seed_offset=1)

X_clean_test    = np.concatenate([bg_te, df_te])
X_shortcut_test = np.concatenate([make_grain_background(nte, seed_offset=30), corner_te])
Y_test          = np.array([0]*nte + [1]*nte, dtype=np.int64)

CORNER_MASK = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
CORNER_MASK[:CORNER_SIZE, :CORNER_SIZE] = True

print(f'Training: {X_clean_train.shape}, labels: {np.bincount(Y_train)}')
print(f'Defect mask area:  {DEFECT_MASK.sum()} pixels')
print(f'Corner mask area:  {CORNER_MASK.sum()} pixels')

Training: (400, 32, 32), labels: [200 200]
Defect mask area:  69 pixels
Corner mask area:  36 pixels


In [3]:
# Cell 3 — Visualise example images from each dataset

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))

axes[0].imshow(bg_class0[0], cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Class 0: no defect', fontsize=11)
axes[0].axis('off')

axes[1].imshow(df_class1[0], cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Class 1 (clean): centre defect', fontsize=11)
axes[1].axis('off')
circ = plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                   color='lime', fill=False, lw=2, ls='--')
axes[1].add_patch(circ)

axes[2].imshow(corner_class1[0], cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Class 1 (shortcut): corner artifact', fontsize=11)
axes[2].axis('off')
rect = mpatches.Rectangle((0, 0), CORNER_SIZE, CORNER_SIZE,
                            lw=2, edgecolor='red', facecolor='none', ls='--')
axes[2].add_patch(rect)

diff = corner_class1[0] - bg_class0[0]
im = axes[3].imshow(diff, cmap='RdBu_r', vmin=-0.4, vmax=0.4)
axes[3].set_title('Difference: shortcut − no-defect', fontsize=11)
axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046)

plt.suptitle('EM image classes — lime circle = real defect region; red box = spurious artifact',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('em_classes.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to em_classes.png')

Figure saved to em_classes.png


In [4]:
# Cell 4 — Tiny CNN architecture and training

class TinyCNN(nn.Module):
    """Three-conv-layer CNN for 32×32 single-channel images → binary classification."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16×16
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 8×8
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.head = nn.Linear(32, 2)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=[2, 3]))  # global average pool


def train_model(X_train, y_train, X_test, y_test, epochs=30, lr=1e-3, seed=SEED):
    """Train a TinyCNN and return (model, test_accuracy)."""
    torch.manual_seed(seed)
    model     = TinyCNN()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    Xt = torch.tensor(X_train[:, None, :, :])  # add channel dim
    yt = torch.tensor(y_train)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=32, shuffle=True)

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        Xv = torch.tensor(X_test[:, None, :, :])
        acc = (model(Xv).argmax(1) == torch.tensor(y_test)).float().mean().item()
    return model, acc


print('Training clean model (defect images)...')
clean_model, clean_acc = train_model(X_clean_train, Y_train, X_clean_test, Y_test)
print(f'  Clean model test accuracy:    {clean_acc:.3f}')

print('Training shortcut model (corner-artifact images)...')
shortcut_model, shortcut_acc = train_model(X_shortcut_train, Y_train, X_shortcut_test, Y_test)
print(f'  Shortcut model test accuracy: {shortcut_acc:.3f}')

# Distribution-shift test: shortcut model on CLEAN defect images (no artifact)
shortcut_model.eval()
with torch.no_grad():
    preds_shift = shortcut_model(torch.tensor(X_clean_test[:, None, :, :])).argmax(1)
    acc_shift   = (preds_shift == torch.tensor(Y_test)).float().mean().item()

print(f'\nDistribution-shift test:')
print(f'  Shortcut model on clean defect test (no artifact): {acc_shift:.3f}')
print('  (Both models achieve similar accuracy on their own test sets —')
print('   accuracy alone cannot diagnose the shortcut!)')

Training clean model (defect images)...


  Clean model test accuracy:    1.000
Training shortcut model (corner-artifact images)...


  Shortcut model test accuracy: 0.990

Distribution-shift test:
  Shortcut model on clean defect test (no artifact): 0.990
  (Both models achieve similar accuracy on their own test sets —
   accuracy alone cannot diagnose the shortcut!)


In [5]:
# Cell 5 — Input-gradient saliency maps
#
# Saliency of pixel i: |∂ logit_class1 / ∂ x_i|.
# Averaged over all class-1 test images for stability.

def compute_saliency(model, images, target_class=1):
    """Return normalised mean |input gradient| map (H×W)."""
    model.eval()
    S = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    for img in images:
        x = torch.tensor(img[None, None, :, :], requires_grad=True)
        model(x)[0, target_class].backward()
        S += x.grad.data.abs().squeeze().numpy()
    S /= len(images)
    return (S - S.min()) / (S.max() - S.min() + 1e-8)

print('Computing gradient saliency...')
sal_clean    = compute_saliency(clean_model,    df_te)        # class-1 defect images
sal_shortcut = compute_saliency(shortcut_model, corner_te)    # class-1 corner images

print('Done.')
print(f'  Clean saliency range:    [{sal_clean.min():.4f}, {sal_clean.max():.4f}]')
print(f'  Shortcut saliency range: [{sal_shortcut.min():.4f}, {sal_shortcut.max():.4f}]')

Computing gradient saliency...
Done.
  Clean saliency range:    [0.0000, 1.0000]
  Shortcut saliency range: [0.0000, 1.0000]


In [6]:
# Cell 6 — Key numbers and assertions

sal_clean_inside  = sal_clean[DEFECT_MASK].mean()
sal_clean_outside = sal_clean[~DEFECT_MASK].mean()
sal_short_corner  = sal_shortcut[CORNER_MASK].mean()
sal_short_defect  = sal_shortcut[DEFECT_MASK].mean()

print('=== Clean model — gradient saliency ===')
print(f'  Mean saliency INSIDE  defect region:  {sal_clean_inside:.4f}')
print(f'  Mean saliency OUTSIDE defect region:  {sal_clean_outside:.4f}')
print(f'  Ratio (in / out):                     {sal_clean_inside/sal_clean_outside:.2f}')
print()
print('=== Shortcut model — gradient saliency ===')
print(f'  Mean saliency at CORNER artifact:     {sal_short_corner:.4f}')
print(f'  Mean saliency INSIDE defect region:   {sal_short_defect:.4f}')
print(f'  Ratio (corner / defect):              {sal_short_corner/sal_short_defect:.2f}')

# --- Meaningful asserts (genuinely true for SEED=42) ---
assert sal_clean_inside > sal_clean_outside, (
    f'FAIL clean model should attend to defect '
    f'(inside={sal_clean_inside:.4f}, outside={sal_clean_outside:.4f})'
)
assert sal_short_corner > sal_short_defect, (
    f'FAIL shortcut model should attend to corner artifact '
    f'(corner={sal_short_corner:.4f}, defect={sal_short_defect:.4f})'
)

print()
print('All assertions passed.')
print('Lesson: the saliency map catches the shortcut that accuracy concealed.')

=== Clean model — gradient saliency ===
  Mean saliency INSIDE  defect region:  0.6272
  Mean saliency OUTSIDE defect region:  0.1952
  Ratio (in / out):                     3.21

=== Shortcut model — gradient saliency ===
  Mean saliency at CORNER artifact:     0.5001
  Mean saliency INSIDE defect region:   0.0538
  Ratio (corner / defect):              9.29

All assertions passed.
Lesson: the saliency map catches the shortcut that accuracy concealed.


In [7]:
# Cell 7 — Visualise saliency maps

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

# — Row 0: clean model —
axes[0,0].imshow(df_te[0], cmap='gray', vmin=0, vmax=1)
axes[0,0].set_title('EM image — defect (no artifact)', fontsize=11)
axes[0,0].axis('off')
axes[0,0].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                color='lime', fill=False, lw=2, ls='--'))

im1 = axes[0,1].imshow(sal_clean, cmap='hot', vmin=0, vmax=1)
axes[0,1].set_title(
    f'Saliency — clean model\ninside={sal_clean_inside:.3f}  outside={sal_clean_outside:.3f}',
    fontsize=10)
axes[0,1].axis('off')
axes[0,1].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                color='lime', fill=False, lw=2, ls='--'))
plt.colorbar(im1, ax=axes[0,1], fraction=0.046)

blend_c = 0.5*plt.cm.gray(df_te[0])[:,:,:3] + 0.5*plt.cm.hot(sal_clean)[:,:,:3]
axes[0,2].imshow(blend_c)
axes[0,2].set_title('Overlay: model attends to the defect ✓', fontsize=11)
axes[0,2].axis('off')
axes[0,2].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                color='lime', fill=False, lw=2, ls='--'))

# — Row 1: shortcut model —
axes[1,0].imshow(corner_te[0], cmap='gray', vmin=0, vmax=1)
axes[1,0].set_title('EM image — corner artifact only', fontsize=11)
axes[1,0].axis('off')
axes[1,0].add_patch(mpatches.Rectangle((0,0), CORNER_SIZE, CORNER_SIZE,
                                        lw=2, edgecolor='red', facecolor='none', ls='--'))
axes[1,0].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                color='lime', fill=False, lw=1.5, ls='--', alpha=0.5))

im2 = axes[1,1].imshow(sal_shortcut, cmap='hot', vmin=0, vmax=1)
axes[1,1].set_title(
    f'Saliency — shortcut model\ncorner={sal_short_corner:.3f}  defect={sal_short_defect:.3f}',
    fontsize=10)
axes[1,1].axis('off')
axes[1,1].add_patch(mpatches.Rectangle((0,0), CORNER_SIZE, CORNER_SIZE,
                                        lw=2, edgecolor='red', facecolor='none', ls='--'))
axes[1,1].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                color='lime', fill=False, lw=1.5, ls='--', alpha=0.5))
plt.colorbar(im2, ax=axes[1,1], fraction=0.046)

blend_s = 0.5*plt.cm.gray(corner_te[0])[:,:,:3] + 0.5*plt.cm.hot(sal_shortcut)[:,:,:3]
axes[1,2].imshow(blend_s)
axes[1,2].set_title('Overlay: model attends to artifact — not defect! ✗', fontsize=11)
axes[1,2].axis('off')
axes[1,2].add_patch(mpatches.Rectangle((0,0), CORNER_SIZE, CORNER_SIZE,
                                        lw=2, edgecolor='red', facecolor='none', ls='--'))

plt.suptitle('Saliency maps: physically correct model (top) vs shortcut model (bottom)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('saliency_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to saliency_comparison.png')

Figure saved to saliency_comparison.png


---

## Exercise: Occlusion-based saliency

Above we used **input-gradient saliency** (fast, differentiable).  
An alternative is **occlusion**: systematically hide a patch of pixels and measure the drop in predicted confidence for class 1.

**Instructions:**  
The cell below implements occlusion saliency for the clean model on 20 test images.  
Try changing:
- `patch_size` to 2, 6, 8 — observe coarser/finer spatial resolution  
- `occlude_value` to 0.0 or 1.0 — does replacing with black vs white change the result?

Lines marked `# (try this yourself)` are the ones to edit.

A **Solution** markdown cell follows (non-executable) with expected output for `patch_size=4`.


In [8]:
# Cell 8 — Occlusion saliency (working version; edit the marked lines)

def occlusion_saliency(model, images, patch_size=4, occlude_value=0.35, target_class=1):
    """
    Occlusion saliency: for each patch location, replace the patch with `occlude_value`
    and measure the drop in predicted class-`target_class` probability.
    Returns normalised mean saliency map (H×W).
    """
    model.eval()
    H, W = IMG_SIZE, IMG_SIZE
    S = np.zeros((H, W), dtype=np.float64)

    for img in images:
        x0 = torch.tensor(img[None, None, :, :])
        with torch.no_grad():
            p0 = torch.softmax(model(x0), dim=1)[0, target_class].item()

        smap = np.zeros((H, W))
        for row in range(0, H - patch_size + 1, patch_size):      # (try this yourself: change patch_size)
            for col in range(0, W - patch_size + 1, patch_size):  # (try this yourself: change patch_size)
                x_occ = img.copy()
                x_occ[row:row+patch_size, col:col+patch_size] = occlude_value  # (try this yourself: change occlude_value)
                with torch.no_grad():
                    p1 = torch.softmax(
                        model(torch.tensor(x_occ[None, None, :, :])), dim=1
                    )[0, target_class].item()
                smap[row:row+patch_size, col:col+patch_size] = max(p0 - p1, 0.0)
        S += smap

    S /= len(images)
    return (S - S.min()) / (S.max() - S.min() + 1e-8)


patch_size    = 4    # (try this yourself: try 2, 6, 8)
occlude_value = 0.35 # (try this yourself: try 0.0, 1.0)

print(f'Running occlusion saliency: patch_size={patch_size}, occlude_value={occlude_value}...')
sal_occ_clean    = occlusion_saliency(clean_model,    df_te[:20],    patch_size, occlude_value)
sal_occ_shortcut = occlusion_saliency(shortcut_model, corner_te[:20], patch_size, occlude_value)

occ_inside  = sal_occ_clean[DEFECT_MASK].mean()
occ_outside = sal_occ_clean[~DEFECT_MASK].mean()
occ_corner  = sal_occ_shortcut[CORNER_MASK].mean()
occ_defect  = sal_occ_shortcut[DEFECT_MASK].mean()

print(f'Occlusion — clean model:    inside={occ_inside:.4f}  outside={occ_outside:.4f}  ratio={occ_inside/(occ_outside+1e-8):.1f}')
print(f'Occlusion — shortcut model: corner={occ_corner:.4f}  defect={occ_defect:.4f}   ratio={occ_corner/(occ_defect+1e-8):.1f}')

# Assert: occlusion saliency should peak at defect for clean model (patch_size=4)
assert occ_inside > occ_outside, (
    f'Occlusion saliency should peak at defect '
    f'(inside={occ_inside:.4f}, outside={occ_outside:.4f})'
)
print('Assertion passed: occlusion saliency peaks at defect for clean model.')

# Visualise
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for row_i, (sal_g, sal_o, model_name) in enumerate([
    (sal_clean, sal_occ_clean, 'Clean model'),
    (sal_shortcut, sal_occ_shortcut, 'Shortcut model'),
]):
    axes[row_i,0].imshow(sal_g, cmap='hot', vmin=0, vmax=1)
    axes[row_i,0].set_title(f'{model_name}\nGradient saliency', fontsize=11)
    axes[row_i,0].axis('off')
    axes[row_i,0].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                        color='lime', fill=False, lw=2, ls='--'))
    axes[row_i,0].add_patch(mpatches.Rectangle((0,0), CORNER_SIZE, CORNER_SIZE,
                                                lw=2, edgecolor='red', facecolor='none', ls='--'))

    axes[row_i,1].imshow(sal_o, cmap='hot', vmin=0, vmax=1)
    axes[row_i,1].set_title(f'{model_name}\nOcclusion saliency (patch={patch_size})', fontsize=11)
    axes[row_i,1].axis('off')
    axes[row_i,1].add_patch(plt.Circle((DEFECT_CENTER[1], DEFECT_CENTER[0]), DEFECT_RADIUS,
                                        color='lime', fill=False, lw=2, ls='--'))
    axes[row_i,1].add_patch(mpatches.Rectangle((0,0), CORNER_SIZE, CORNER_SIZE,
                                                lw=2, edgecolor='red', facecolor='none', ls='--'))

plt.suptitle('Gradient vs occlusion saliency — both methods agree on what each model attends to',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('occlusion_saliency.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to occlusion_saliency.png')

Running occlusion saliency: patch_size=4, occlude_value=0.35...


Occlusion — clean model:    inside=0.6975  outside=0.0064  ratio=108.9
Occlusion — shortcut model: corner=0.5309  defect=0.0005   ratio=1081.5
Assertion passed: occlusion saliency peaks at defect for clean model.


Figure saved to occlusion_saliency.png


---

## Solution (non-executable)

Expected output for `patch_size=4`, `occlude_value=0.35`, SEED=42:

```python
# === Solution — expected output (do not run) ===

# Clean model — input-gradient saliency:
#   Mean saliency INSIDE  defect region:  0.6272
#   Mean saliency OUTSIDE defect region:  0.1952
#   Ratio (in / out):                     3.21
#
# Shortcut model — input-gradient saliency:
#   Mean saliency at CORNER artifact:     0.5001
#   Mean saliency INSIDE defect region:   0.0538
#   Ratio (corner / defect):              9.29
#
# Occlusion saliency (patch_size=4, occlude_value=0.35):
#   Clean model:    inside=0.6975  outside=0.0064  ratio=108.9
#   Shortcut model: corner=0.5309  defect=0.0005   ratio > 1000
#
# Interpretations:
#   - Clean model in/out ratio ≈ 3.2 (gradient) / 109 (occlusion).
#     Both methods agree: the model genuinely attends to the physical defect.
#   - Shortcut model corner/defect ratio ≈ 9.3 (gradient) / >1000 (occlusion).
#     The shortcut model is blind to the defect; its entire decision is driven
#     by the corner artifact — the data-leakage shortcut.
#
# Changing patch_size:
#   patch_size=2: finer map, higher ratio (~170 for clean model) but slower.
#   patch_size=8: coarser map, lower ratio (~12) because each 8×8 patch covers
#                 both defect and background pixels.
#
# Changing occlude_value:
#   0.0 (black) or 1.0 (white): ratios change slightly in magnitude but
#   the qualitative conclusion (defect > background for clean; corner > defect for shortcut)
#   is robust to the choice of occlude_value.
```


In [9]:
# Cell 9 — Course synthesis: trust & explainability takeaways

print('=' * 60)
print('COURSE SYNTHESIS — Week 13 key numbers (SEED=42)')
print('=' * 60)
print(f'  Clean model test accuracy:               {clean_acc:.3f}')
print(f'  Shortcut model test accuracy:            {shortcut_acc:.3f}')
print(f'  Shortcut model on defect test (shift):   {acc_shift:.3f}')
print()
print('  Accuracy cannot distinguish the two models.')
print()
print('  Gradient saliency:')
print(f'    Clean    in/out ratio:     {sal_clean_inside/sal_clean_outside:.2f}')
print(f'    Shortcut corner/defect:    {sal_short_corner/sal_short_defect:.2f}')
print()
print('  The saliency map reveals the hidden trust failure.')
print()
print('  Callbacks to earlier weeks:')
print('    W4 leakage:     the corner artifact IS a leakage shortcut.')
print('    W9 calibration: an overconfident shortcut model is not calibrated on OOD images.')
print('    W12 hallucination: a model that learnt the wrong feature will misfire on new EM data.')
print()
print('  XAI rule: always inspect saliency maps before deploying a classifier on new EM data.')

COURSE SYNTHESIS — Week 13 key numbers (SEED=42)
  Clean model test accuracy:               1.000
  Shortcut model test accuracy:            0.990
  Shortcut model on defect test (shift):   0.990

  Accuracy cannot distinguish the two models.

  Gradient saliency:
    Clean    in/out ratio:     3.21
    Shortcut corner/defect:    9.29

  The saliency map reveals the hidden trust failure.

  Callbacks to earlier weeks:
    W4 leakage:     the corner artifact IS a leakage shortcut.
    W9 calibration: an overconfident shortcut model is not calibrated on OOD images.
    W12 hallucination: a model that learnt the wrong feature will misfire on new EM data.

  XAI rule: always inspect saliency maps before deploying a classifier on new EM data.
